# Asistente experto en teledetección hiperespectral agrícola
### Proyecto final — IA Generativa: Gemini + RAG (ChromaDB) + agente LangGraph con memoria

**Dominio.** Documentación técnica: cinco artículos científicos (MDPI *Sensors* y *Frontiers in Plant Science*) sobre el uso de índices de vegetación hiperespectrales para detectar estrés hídrico, plagas y enfermedades en cultivos (arroz, tomate, trigo, *Brassica*, reconocimiento de cultivos). Son 82 páginas de texto técnico con siglas, cifras y nombres de índices, un buen banco de pruebas para un RAG que debe citar con precisión.

**Stack.** Gemini (`gemini-embedding-001` para embeddings, `gemini-3.1-flash-lite` como LLM; el plan gratuito limita `gemini-3.5-flash` a 20 peticiones/día, configurable con `LLM_MODEL`) · ChromaDB persistente · LangGraph sobre LangChain · memoria de conversación con `MemorySaver` · Streamlit opcional (`app.py`).

**Qué se demuestra en este notebook**
1. Construcción y verificación de la base de conocimiento vectorial.
2. System prompt y su justificación (resumen; completa en el README).
3. Agente en LangGraph: `condense → retrieve → generate`.
4. Memoria de conversación: preguntas que se refieren a respuestas anteriores.
5. Seis preguntas de ejemplo documentadas.
6. Celda de chat interactiva.

## 0. Preparación
La clave de Gemini se lee de `.env` (`GOOGLE_API_KEY`), nunca del código. El código del proyecto vive en `src/` para que el notebook, la app Streamlit y los tests compartan la misma implementación.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # el notebook está en notebooks/

import os
from src import config
assert os.getenv("GOOGLE_API_KEY"), "Falta GOOGLE_API_KEY: copia .env.example a .env y escribe tu clave"
print("Embeddings:", config.EMBEDDING_MODEL, f"({config.EMBEDDING_DIM} dims)")
print("LLM:       ", config.LLM_MODEL)
print("Chroma:    ", config.CHROMA_DIR)

Embeddings: models/gemini-embedding-001 (768 dims)
LLM:        gemini-3.1-flash-lite
Chroma:     /Users/ayahibatallahaichoune/prueba_Ai/proyecto-final-genai/chroma_db


## 1. Base de conocimiento vectorial (ChromaDB + Gemini Embeddings)

**Preprocesado** (`src/ingest.py`): extracción por página con `pypdf`, unión de palabras partidas por guion, limpieza de saltos de línea, y troceado en chunks de ~1.200 caracteres con solapamiento de 200, cortando preferentemente en fin de frase y **sin cruzar páginas**. Cada chunk guarda `source`, `title` y `page`, lo que permite al agente citar documento y página exactos.

**Indexación** (`src/knowledge_base.py`): colección `papers_teledeteccion` en ChromaDB persistente (`chroma_db/`), distancia coseno, embeddings de `gemini-embedding-001` a 768 dimensiones. Los chunks se envían en lotes de 25 con reintentos, porque el plan gratuito de Gemini limita las peticiones por minuto. Si la colección ya existe, se reutiliza.

In [2]:
from src.ingest import load_documents

docs = load_documents()
por_doc = {}
for d in docs:
    m = d.metadata
    por_doc.setdefault(m["source"], {"título": m["title"][:70], "páginas": 0, "chunks": 0})
    por_doc[m["source"]]["chunks"] += 1
    por_doc[m["source"]]["páginas"] = max(por_doc[m["source"]]["páginas"], m["page"])

print(f"{len(docs)} chunks en total\n")
for src, info in por_doc.items():
    print(f"{src:24s} {info['páginas']:3d} pág. {info['chunks']:4d} chunks  {info['título']}")
print("\nEjemplo de chunk:\n", docs[40].metadata, "\n", docs[40].page_content[:400], "...")

352 chunks en total

fpls-08-01219.pdf         12 pág.   67 chunks  Canopy Vegetation Indices from In situ Hyperspectral Data to Assess Pl
sensors-19-00375.pdf      12 pág.   43 chunks  Sensitivity of a Ratio Vegetation Index Derived from Hyperspectral Rem
sensors-21-05705.pdf      17 pág.   75 chunks  Ground-Based Hyperspectral Remote Sensing for Estimating Water Stress 
sensors-24-01916.pdf      15 pág.   81 chunks  Investigation of Using Hyperspectral Vegetation Indices to Assess Bras
sensors-25-03817.pdf      26 pág.   86 chunks  NDVI and Beyond: Vegetation Indices as Features for Crop Recognition a

Ejemplo de chunk:
 {'source': 'fpls-08-01219.pdf', 'title': 'Canopy Vegetation Indices from In situ Hyperspectral Data to Assess Plant Water Status of Winter Wheat under Powdery Mildew Stress', 'page': 9, 'chunk_id': 'fpls-08-01219:p9:c0'} 
 Feng et al. PRI for Water Content Assessment FIGURE 8 | Comparison of predicted and observed wheat PWC based on the PRI in winter wheat. Data poin

In [3]:
from src.knowledge_base import build_knowledge_base, describe_collection, retrieve

store = build_knowledge_base()        # reutiliza chroma_db/ si ya está construida
describe_collection(store)["n_chunks"]

Colección 'papers_teledeteccion' ya existe con 352 chunks; se reutiliza.


352

### 1.1 Verificación de la colección antes de conectar el agente
Tres consultas de prueba, en español e inglés, sobre documentos distintos. Se comprueba que el documento esperado aparece en primera posición y que la similitud coseno es claramente mayor que la de una consulta fuera del dominio.

In [4]:
pruebas = [
    ("¿Qué índice detecta el daño del saltahojas marrón en arroz?", "sensors-19-00375.pdf"),
    ("water stress indices in greenhouse tomato", "sensors-21-05705.pdf"),
    ("¿Cómo se evalúa el mildiu en Brassica con índices hiperespectrales?", "sensors-24-01916.pdf"),
    ("receta de paella valenciana", None),   # fuera del dominio: similitud baja esperada
]
for q, esperado in pruebas:
    hits = retrieve(store, q, k=3)
    top = hits[0][0].metadata
    ok = "✓" if esperado is None or top["source"] == esperado else "✗"
    print(f"{ok} {q}")
    for d, sim in hits:
        print(f"     sim={sim:.3f}  {d.metadata['source']} p.{d.metadata['page']}  {d.page_content[:70]}...")

✓ ¿Qué índice detecta el daño del saltahojas marrón en arroz?
     sim=0.760  sensors-19-00375.pdf p.1  emote sensing end-products related to vegetation have potential applic...
     sim=0.753  sensors-19-00375.pdf p.1  f BPH. The relationships between RVI746/670, the number of BPH and exp...
     sim=0.749  sensors-19-00375.pdf p.9  o BPHs for four and six days, it signiﬁcantly reduced two to six days ...


✓ water stress indices in greenhouse tomato
     sim=0.792  sensors-21-05705.pdf p.13  l liquid water in leaves instead of vapor. The results are in agreemen...
     sim=0.787  sensors-21-05705.pdf p.3  water loss [4], or leaching of agricultural nutrients that would degra...
     sim=0.774  sensors-21-05705.pdf p.13  tatus with remote sensing from the spatial and temporal resolutions it...


✓ ¿Cómo se evalúa el mildiu en Brassica con índices hiperespectrales?
     sim=0.829  sensors-24-01916.pdf p.1  wavelengths that could be used to develop downy mildew indices (DMI). ...
     sim=0.820  sensors-24-01916.pdf p.3  studies have been conducted to evaluate young diseased Brassica with v...
     sim=0.819  sensors-24-01916.pdf p.1  hnic State University, San Luis Obispo, CA 93407, USA; mferna36@calpol...


✓ receta de paella valenciana
     sim=0.556  sensors-24-01916.pdf p.15  Ref] Disclaimer/Publisher’s Note: The statements, opinions and data co...
     sim=0.523  sensors-19-00375.pdf p.1  sensors Article Sensitivity of a Ratio Vegetation Index Derived from H...
     sim=0.520  sensors-19-00375.pdf p.1  emote sensing end-products related to vegetation have potential applic...


## 2. System prompt

El prompt completo está en `src/agent.py` (`SYSTEM_PROMPT`). Las decisiones, resumidas:

| Decisión | Por qué |
|---|---|
| Rol acotado: experto en teledetección hiperespectral agrícola | Un rol concreto reduce respuestas genéricas y fija el vocabulario técnico. |
| Responder **solo** con los fragmentos y abstenerse explícitamente si no bastan | Es la regla que evita alucinaciones; se comprueba con una pregunta fuera del corpus. |
| Citas `[n]` tras cada afirmación | Hace la respuesta verificable: cada cita se corresponde con un documento y una página mostrados al usuario. |
| Responder en el idioma de la pregunta | Los artículos están en inglés y el usuario puede preguntar en español. |
| Resolver referencias al historial ("ese índice") | Es lo que convierte el RAG en una conversación con memoria. |
| Dos a seis frases, tono técnico, sin relleno | Respuestas rápidas de leer; la temperatura baja (0,1) y `thinking_budget=0` van en la misma línea. |

La justificación completa está en el README.

In [5]:
from src.agent import SYSTEM_PROMPT, CONDENSE_PROMPT
print(SYSTEM_PROMPT)
print("\n--- prompt de reescritura de la pregunta (memoria aplicada a la recuperación) ---\n")
print(CONDENSE_PROMPT)

Eres un asistente experto en teledetección hiperespectral aplicada a la agricultura (índices de vegetación, estrés hídrico, plagas y enfermedades de cultivos). Tu conocimiento proviene EXCLUSIVAMENTE de los fragmentos de artículos científicos que se te proporcionan en cada turno.

Reglas:
1. Responde solo con información contenida en los fragmentos. Si no contienen lo necesario, dilo claramente ("Los documentos no contienen información sobre...") y no inventes ni completes con conocimiento general.
2. Cita el fragmento que sostiene cada afirmación con el formato [n] (por ejemplo [2]); puedes citar varios: [1][3].
3. Responde en el mismo idioma en que está escrita la pregunta.
4. Mantén el hilo de la conversación: si la pregunta se refiere a algo mencionado antes ("ese índice", "el mismo cultivo"), resuélvelo con el historial. Si hace referencia a algo que no aparece ni en el historial ni en los fragmentos, no supongas a qué se refiere: pide que lo concrete.
5. Tono técnico y preciso, s

## 3. Agente en LangGraph

Grafo `START → condense → retrieve → generate → END`:

- **condense**: si hay historial, Gemini reescribe la última pregunta como una pregunta autónoma (resuelve "ese índice", "el mismo cultivo"). Así la memoria mejora también la recuperación, no solo la redacción.
- **retrieve**: top-5 chunks de ChromaDB para la pregunta reescrita.
- **generate**: Gemini (`gemini-3.1-flash-lite`, temperatura 0,1, sin razonamiento extendido) responde con el system prompt, los fragmentos numerados y el historial recortado a los últimos 5 turnos.

La memoria la aporta el *checkpointer* `MemorySaver`: el estado (mensajes incluidos) se guarda por `thread_id`, de modo que cada hilo es una conversación independiente.

In [6]:
from src.agent import build_agent, ask, history

agent = build_agent(store)
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	condense(condense)
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> condense;
	condense --> retrieve;
	retrieve --> generate;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [7]:
r = ask(agent, "¿Qué índice de vegetación es más sensible al daño del saltahojas marrón en arroz?", thread_id="demo")
print(r["answer"])
print("\nFuentes:")
for s in r["sources"]:
    print(f"  [{s['n']}] {s['source']} p.{s['page']}  sim={s['score']}")

El índice de vegetación RVI746/670 es sensible para identificar el daño causado por el saltahojas marrón (BPH) en plantas de arroz [1][4]. Este índice permite detectar tanto la compensación fisiológica inicial de la planta ante una infestación leve como el efecto posterior derivado del daño por la alimentación de los insectos [3][4][5].

Fuentes:
  [1] sensors-19-00375.pdf p.1  sim=0.792
  [2] sensors-19-00375.pdf p.1  sim=0.783
  [3] sensors-19-00375.pdf p.9  sim=0.782
  [4] sensors-19-00375.pdf p.1  sim=0.781
  [5] sensors-19-00375.pdf p.9  sim=0.779


## 4. Memoria de conversación

La segunda pregunta no menciona el índice ni el cultivo: solo tiene sentido con la respuesta anterior. Se muestra la pregunta reescrita por el nodo `condense` para ver cómo el agente usa el historial, y después que un hilo distinto no comparte esa memoria.

In [8]:
r2 = ask(agent, "¿Y en qué bandas se basa ese índice?", thread_id="demo")
print("Pregunta original :", r2["question"])
print("Pregunta reescrita:", r2["standalone_question"])
print("\nRespuesta:", r2["answer"])
print("\nMensajes en la memoria del hilo 'demo':", len(history(agent, "demo")))

Pregunta original : ¿Y en qué bandas se basa ese índice?
Pregunta reescrita: ¿En qué bandas espectrales se basa el índice de vegetación RVI746/670 utilizado para detectar el daño del saltahojas marrón en el cultivo de arroz?

Respuesta: El índice RVI746/670 se basa en las bandas espectrales de 746 nm (infrarrojo cercano) y 670 nm (rojo) [1].

Mensajes en la memoria del hilo 'demo': 4


In [9]:
r3 = ask(agent, "¿Y cuántas plantas por maceta usaron en ese experimento?", thread_id="demo")
print("Pregunta reescrita:", r3["standalone_question"])
print("\nRespuesta:", r3["answer"])

Pregunta reescrita: ¿Cuántas plantas de arroz por maceta se utilizaron en el experimento para evaluar la sensibilidad del índice de vegetación RVI746/670 ante el daño del saltahojas marrón?

Respuesta: Los documentos proporcionados no contienen información sobre el número de plantas por maceta utilizadas en el experimento. Solo se especifica que se utilizaron plantas de arroz en etapa de macollamiento (tillering stage) [5].


In [10]:
# Un hilo nuevo no hereda el contexto: la misma pregunta ambigua no puede resolverse
# y el agente debe pedir que se concrete en lugar de suponer un antecedente
r_otro = ask(agent, "¿Y en qué bandas se basa ese índice?", thread_id="otro_hilo")
print("Pregunta reescrita:", r_otro["standalone_question"])
print("\nRespuesta:", r_otro["answer"])
print("\nMensajes en 'otro_hilo':", len(history(agent, "otro_hilo")), "| en 'demo':", len(history(agent, "demo")))

Pregunta reescrita: ¿Y en qué bandas se basa ese índice?

Respuesta: Para responder a tu pregunta, necesito que especifiques a qué índice te refieres, ya que en el historial de la conversación no se ha mencionado ninguno en particular. Los documentos proporcionados contienen fórmulas para múltiples índices, como el mCARI, MTVI2, NDVI 750, RVSI, WI, GEMI o GARI, entre otros [1][2][4]. Por favor, indica el nombre del índice sobre el cual deseas conocer las bandas espectrales utilizadas.

Mensajes en 'otro_hilo': 2 | en 'demo': 6


## 5. Preguntas de ejemplo documentadas
Seis preguntas en un hilo nuevo: una por documento, una en inglés y una fuera del corpus para comprobar la abstención.

In [11]:
ejemplos = [
    "¿Qué índices hiperespectrales permiten evaluar la gravedad del mildiu (downy mildew) en Brassica y con qué precisión?",
    "Which vegetation indices correlate best with water stress in tomato, and which sensor was used?",
    "¿Es suficiente el NDVI como característica para reconocer y segmentar cultivos en imágenes hiperespectrales?",
    "¿Cómo se evalúa el estado hídrico del trigo de invierno bajo estrés por oídio con datos hiperespectrales in situ?",
    "¿Qué rango de longitudes de onda medía el espectrómetro usado en el estudio del tomate?",
    "¿Qué dicen los documentos sobre redes LoRaWAN para sensores agrícolas?",
]
resultados = []
for i, q in enumerate(ejemplos, 1):
    time.sleep(4)         # margen para la cuota por minuto del plan gratuito
    t0 = time.time()
    r = ask(agent, q, thread_id="ejemplos")
    resultados.append(r)
    print(f"\n{'='*100}\n{i}. {q}   ({time.time()-t0:.1f} s)\n")
    print(r["answer"])
    print("\nFuentes citadas:")
    citadas = {int(n) for n in __import__('re').findall(r'\[(\d+)\]', r['answer'])}
    for s in r["sources"]:
        marca = "*" if s["n"] in citadas else " "
        print(f"  {marca}[{s['n']}] {s['source']} p.{s['page']}  sim={s['score']}")


1. ¿Qué índices hiperespectrales permiten evaluar la gravedad del mildiu (downy mildew) en Brassica y con qué precisión?   (1.1 s)

Para evaluar los niveles de infección por mildiu (DM) en *Brassica*, se evaluaron 37 índices de vegetación existentes y tres índices de mildiu (DMI) propuestos [1][3]. Utilizando un modelo de máquina de vectores de soporte (SVM) y los DMI propuestos, se alcanzó una precisión del 71,3% para DM1 (infección temprana), 80,7% para DM2 (infección progresada) y 85,3% para DM3 (infección severa) al distinguir estas de hojas sanas [1].

Fuentes citadas:
  *[1] sensors-24-01916.pdf p.1  sim=0.842
   [2] sensors-24-01916.pdf p.3  sim=0.824
  *[3] sensors-24-01916.pdf p.1  sim=0.823
   [4] sensors-24-01916.pdf p.3  sim=0.808
   [5] sensors-24-01916.pdf p.1  sim=0.806



2. Which vegetation indices correlate best with water stress in tomato, and which sensor was used?   (1.5 s)

The vegetation indices that correlate best with water stress in tomato are NDVI, RDVI, WI, NDWI, NDWI1640, PRI570, and PRInorm [2][3]. These indices were evaluated using ASD hyperspectral reflectance data [2].

Fuentes citadas:
   [1] sensors-21-05705.pdf p.13  sim=0.793
  *[2] sensors-21-05705.pdf p.1  sim=0.787
  *[3] sensors-21-05705.pdf p.13  sim=0.785
   [4] sensors-21-05705.pdf p.10  sim=0.784
   [5] sensors-21-05705.pdf p.1  sim=0.782



3. ¿Es suficiente el NDVI como característica para reconocer y segmentar cultivos en imágenes hiperespectrales?   (1.9 s)

Los documentos indican que, aunque el NDVI es el índice más popular, existen muchos otros índices de vegetación que presentan diferencias significativas dependiendo del contexto del paisaje y la aplicación [1][4]. El estudio sugiere que, si bien muchos índices tienen correlación con el NDVI, el uso de una variedad de índices es efectivo como características discriminativas para la segmentación y el reconocimiento de cultivos [1]. Por tanto, el uso de múltiples índices, evaluados mediante clasificadores como RF, MLP, SVM y GBM, permite capturar diferencias funcionales y contextuales que apoyan la identificación de cultivos [3][5].

Fuentes citadas:
  *[1] sensors-25-03817.pdf p.1  sim=0.79
   [2] sensors-25-03817.pdf p.1  sim=0.788
  *[3] sensors-25-03817.pdf p.17  sim=0.781
  *[4] sensors-25-03817.pdf p.1  sim=0.774
  *[5] sensors-25-03817.pdf p.2  sim=0.772



4. ¿Cómo se evalúa el estado hídrico del trigo de invierno bajo estrés por oídio con datos hiperespectrales in situ?   (2.0 s)

El estado hídrico del trigo de invierno bajo estrés por oídio se evalúa mediante sensores hiperespectrales *in situ* que detectan el contenido de agua de la planta (PWC) a escala de dosel [1][2][5]. Se utilizan índices de vegetación basados principalmente en bandas verdes, los cuales han mostrado coeficientes de correlación con el PWC de entre 0,784 y 0,902 [1]. Asimismo, el índice de reflectancia fotoquímica (PRI) se recomienda como un indicador fiable, habiendo demostrado un $R^2$ de 0,819 entre los valores medidos y predichos [3]. Por el contrario, el índice de agua (WBI) puede no ser adecuado para detectar la dinámica de humedad bajo estrés biológico, ya que este requiere pigmentos del ciclo de las xantofilas [4].

Fuentes citadas:
  *[1] fpls-08-01219.pdf p.1  sim=0.811
  *[2] fpls-08-01219.pdf p.1  sim=0.797
  *[3] fpls-08-01219.pdf p.1  sim=0.783
  *[4


5. ¿Qué rango de longitudes de onda medía el espectrómetro usado en el estudio del tomate?   (2.0 s)

El espectrómetro hiperespectral ASD Field Spec 4 utilizado en el estudio mide un rango de longitudes de onda comprendido entre los 350 y los 2500 nm [1].

Fuentes citadas:
  *[1] sensors-21-05705.pdf p.6  sim=0.787
   [2] sensors-21-05705.pdf p.6  sim=0.761
   [3] sensors-21-05705.pdf p.9  sim=0.758
   [4] sensors-21-05705.pdf p.1  sim=0.756
   [5] sensors-21-05705.pdf p.1  sim=0.756



6. ¿Qué dicen los documentos sobre redes LoRaWAN para sensores agrícolas?   (1.8 s)

Los documentos proporcionados no contienen información sobre redes LoRaWAN aplicadas a sensores agrícolas.

Fuentes citadas:
   [1] sensors-25-03817.pdf p.24  sim=0.679
   [2] sensors-25-03817.pdf p.25  sim=0.672
   [3] sensors-25-03817.pdf p.24  sim=0.67
   [4] sensors-24-01916.pdf p.14  sim=0.661
   [5] sensors-25-03817.pdf p.25  sim=0.661


### 5.1 Limitación conocida: preguntas comparativas entre documentos

Una pregunta que exige combinar dos artículos ("compara cómo abordan el estrés hídrico el artículo del tomate y el del trigo") se recupera con **una sola consulta** al índice, y los cinco fragmentos más cercanos suelen venir del documento que domina el embedding de la pregunta. El agente no inventa: dice que no tiene información sobre el trigo, lo cual es correcto respecto a los fragmentos que ha recibido, pero no respecto al corpus. La solución sería descomponer la pregunta en sub-consultas (una por documento) o recuperar un cupo de fragmentos por documento; queda como siguiente paso.

In [12]:
r = ask(agent, "Compara brevemente cómo abordan el estrés hídrico el artículo del tomate y el del trigo.", thread_id="limitacion")
print(r["answer"])
print("\nDocumentos de los fragmentos recuperados:", sorted({s["source"] for s in r["sources"]}))

Los documentos proporcionados no contienen información sobre el cultivo de trigo, por lo que no es posible realizar la comparación solicitada. Respecto al tomate, los estudios utilizan índices de reflectancia hiperespectral como NDVI, RDVI, WI, NDWI, NDWI 1640, PRI570 y PRInorm para estimar el estrés hídrico en diferentes etapas de crecimiento y tipos de suelo [4].

Documentos de los fragmentos recuperados: ['sensors-21-05705.pdf']


## 6. Celda de chat interactiva

Escribe una pregunta y pulsa **Enviar** (o Intro). La conversación mantiene memoria en el hilo `chat`; **Nueva conversación** empieza un hilo nuevo. Si los widgets no se muestran (por ejemplo en algunos visores), la celda siguiente ofrece la misma conversación con `input()`.

In [13]:
import uuid
import ipywidgets as w
from IPython.display import display, Markdown

estado = {"thread": "chat"}
entrada = w.Text(placeholder="Pregunta sobre los artículos...", layout=w.Layout(width="70%"))
boton = w.Button(description="Enviar", button_style="primary")
nuevo = w.Button(description="Nueva conversación")
salida = w.Output()

def enviar(_=None):
    q = entrada.value.strip()
    if not q:
        return
    entrada.value = ""
    with salida:
        display(Markdown(f"**Tú:** {q}"))
        r = ask(agent, q, thread_id=estado["thread"])
        fuentes = "; ".join(f"[{s['n']}] {s['source']} p.{s['page']}" for s in r["sources"])
        nota = "" if r["standalone_question"] == q else f"\n\n<sub>Interpretada como: *{r['standalone_question']}*</sub>"
        display(Markdown(f"**Asistente:** {r['answer']}\n\n<sub>Fuentes: {fuentes}</sub>{nota}"))

def reiniciar(_=None):
    estado["thread"] = f"chat-{uuid.uuid4().hex[:6]}"
    salida.clear_output()
    with salida:
        display(Markdown("*Nueva conversación.*"))

boton.on_click(enviar); entrada.on_submit(enviar); nuevo.on_click(reiniciar)
display(w.HBox([entrada, boton, nuevo]), salida)

/var/folders/l3/vnjjpqj969q8c7j0lcjl_0fm0000gp/T/ipykernel_13043/1871144541.py:29: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  boton.on_click(enviar); entrada.on_submit(enviar); nuevo.on_click(reiniciar)


Output()

In [14]:
# Alternativa sin widgets (descomentar para usar): escribe 'salir' para terminar.
# while True:
#     q = input("Tú: ").strip()
#     if q.lower() in {"salir", "exit"}: break
#     r = ask(agent, q, thread_id="chat-input")
#     print("Asistente:", r["answer"], "\n")

## 7. Conclusiones y limitaciones

- La recuperación densa con Gemini Embeddings sitúa el documento correcto en primera posición en todas las consultas de prueba, y el agente se abstiene cuando el corpus no contiene la respuesta.
- La memoria funciona en dos niveles: el historial entra en la generación y, además, reescribe la pregunta antes de recuperar; es lo que resuelve "ese índice" o "ese experimento".
- Limitaciones: la memoria es en proceso (`MemorySaver`), así que se pierde al reiniciar el kernel; para producción se cambiaría por un checkpointer persistente (SQLite/Postgres). No hay reranker ni búsqueda léxica (BM25), que ayudarían con siglas y cifras exactas en corpus mayores. La evaluación es cualitativa; el siguiente paso sería un conjunto de preguntas con respuesta esperada y métricas de recuperación y fidelidad.